# Transfer-based black-box adversarial attack between the dense and pruned models

In [ ]:
import torch
import torch.nn as nn

# Same normalization as your CIFAR test loader
CIFAR_MEAN = torch.tensor([0.485, 0.456, 0.406])
CIFAR_STD = torch.tensor([0.229, 0.224, 0.225])


def get_normalized_bounds(mean, std, device):
    """
    What this does:
    - Converts the valid raw pixel range [0, 1] into normalized-space bounds
    - These bounds are used to clamp adversarial images correctly
    """
    mean = mean.to(device).view(1, 3, 1, 1)
    std = std.to(device).view(1, 3, 1, 1)

    clamp_min = (0.0 - mean) / std
    clamp_max = (1.0 - mean) / std
    return clamp_min, clamp_max


def fgsm_attack_normalized(images, epsilon, data_grad, std):
    """
    What this does:
    - Applies FGSM to normalized images
    - Interprets epsilon in raw pixel space, then rescales it channel-wise
    - Returns adversarial images in normalized space
    """
    std = std.to(images.device).view(1, 3, 1, 1)
    epsilon_normalized = epsilon / std
    adv_images = images + epsilon_normalized * data_grad.sign()
    return adv_images


def generate_transfer_fgsm_batch(source_model, images, labels, epsilon, device):
    """
    What this does:
    - Uses the source model to compute input gradients
    - Creates FGSM adversarial examples from those gradients
    - These adversarial examples can then be tested on another model
    """
    criterion = nn.CrossEntropyLoss()

    mean = CIFAR_MEAN.to(device)
    std = CIFAR_STD.to(device)
    clamp_min, clamp_max = get_normalized_bounds(mean, std, device)

    images = images.to(device).detach().clone()
    labels = labels.to(device)

    images.requires_grad = True

    outputs = source_model(images)
    loss = criterion(outputs, labels)

    source_model.zero_grad()
    loss.backward()

    data_grad = images.grad.detach()
    adv_images = fgsm_attack_normalized(images, epsilon, data_grad, std)

    adv_images = torch.max(torch.min(adv_images, clamp_max), clamp_min)
    adv_images = adv_images.detach()

    return adv_images


def evaluate_transfer_attack(source_model, target_model, testloader, epsilon, device, save_per_batch=0):
    """
    What this does:
    - Creates FGSM adversarial examples on the source model
    - Evaluates those same adversarial examples on the target model
    - Reports:
        1. clean accuracy of source model
        2. clean accuracy of target model
        3. adversarial accuracy on source model
        4. adversarial accuracy on target model
        5. transfer fooling rate on initially correct target examples
    - Optionally saves a few successful transferred examples per batch
    """
    source_model.eval()
    target_model.eval()

    total = 0
    source_clean_correct = 0
    target_clean_correct = 0
    source_adv_correct = 0
    target_adv_correct = 0

    target_initially_correct = 0
    target_transfer_success = 0

    saved_examples = []

    for images, labels in testloader:
        images = images.to(device)
        labels = labels.to(device)

        # Clean predictions
        with torch.no_grad():
            source_clean_outputs = source_model(images)
            target_clean_outputs = target_model(images)

            source_clean_preds = source_clean_outputs.argmax(dim=1)
            target_clean_preds = target_clean_outputs.argmax(dim=1)

        # Generate adversarial examples using SOURCE model gradients
        adv_images = generate_transfer_fgsm_batch(
            source_model=source_model,
            images=images,
            labels=labels,
            epsilon=epsilon,
            device=device
        )

        # Evaluate both models on those same adversarial images
        with torch.no_grad():
            source_adv_outputs = source_model(adv_images)
            target_adv_outputs = target_model(adv_images)

            source_adv_preds = source_adv_outputs.argmax(dim=1)
            target_adv_preds = target_adv_outputs.argmax(dim=1)

        batch_size = labels.size(0)
        total += batch_size

        source_clean_correct += (source_clean_preds == labels).sum().item()
        target_clean_correct += (target_clean_preds == labels).sum().item()

        source_adv_correct += (source_adv_preds == labels).sum().item()
        target_adv_correct += (target_adv_preds == labels).sum().item()

        # Transfer fooling rate: among target-clean-correct examples, how many are flipped by transferred attack?
        target_correct_mask = (target_clean_preds == labels)
        target_transfer_success_mask = target_correct_mask & (target_adv_preds != labels)

        target_initially_correct += target_correct_mask.sum().item()
        target_transfer_success += target_transfer_success_mask.sum().item()

        if save_per_batch > 0:
            idxs = target_transfer_success_mask.nonzero(as_tuple=True)[0][:save_per_batch]
            for idx in idxs:
                saved_examples.append({
                    "clean_image": images[idx].detach().cpu(),
                    "adv_image": adv_images[idx].detach().cpu(),
                    "label": labels[idx].item(),
                    "source_clean_pred": source_clean_preds[idx].item(),
                    "source_adv_pred": source_adv_preds[idx].item(),
                    "target_clean_pred": target_clean_preds[idx].item(),
                    "target_adv_pred": target_adv_preds[idx].item(),
                })

    source_clean_acc = source_clean_correct / total
    target_clean_acc = target_clean_correct / total
    source_adv_acc = source_adv_correct / total
    target_adv_acc = target_adv_correct / total

    target_transfer_fooling_rate = (
        target_transfer_success / target_initially_correct
        if target_initially_correct > 0 else 0.0
    )

    return {
        "source_clean_acc": source_clean_acc,
        "target_clean_acc": target_clean_acc,
        "source_adv_acc": source_adv_acc,
        "target_adv_acc": target_adv_acc,
        "target_transfer_fooling_rate": target_transfer_fooling_rate,
        "saved_examples": saved_examples,
    }

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dense_model = dense_model.to(device)
pruned_model = pruned_model.to(device)

epsilon = 8 / 255

# Dense -> Pruned
dense_to_pruned = evaluate_transfer_attack(
    source_model=dense_model,
    target_model=pruned_model,
    testloader=testloader,
    epsilon=epsilon,
    device=device,
    save_per_batch=3
)

print("Dense -> Pruned transfer")
print(f"Source clean acc: {dense_to_pruned['source_clean_acc']:.4f}")
print(f"Target clean acc: {dense_to_pruned['target_clean_acc']:.4f}")
print(f"Source adv acc:   {dense_to_pruned['source_adv_acc']:.4f}")
print(f"Target adv acc:   {dense_to_pruned['target_adv_acc']:.4f}")
print(f"Transfer fooling rate: {dense_to_pruned['target_transfer_fooling_rate']:.4f}")


# Pruned -> Dense
pruned_to_dense = evaluate_transfer_attack(
    source_model=pruned_model,
    target_model=dense_model,
    testloader=testloader,
    epsilon=epsilon,
    device=device,
    save_per_batch=3
)

print("\nPruned -> Dense transfer")
print(f"Source clean acc: {pruned_to_dense['source_clean_acc']:.4f}")
print(f"Target clean acc: {pruned_to_dense['target_clean_acc']:.4f}")
print(f"Source adv acc:   {pruned_to_dense['source_adv_acc']:.4f}")
print(f"Target adv acc:   {pruned_to_dense['target_adv_acc']:.4f}")
print(f"Transfer fooling rate: {pruned_to_dense['target_transfer_fooling_rate']:.4f}")